# NeuraVol Forecast: neural IV-surface model (Phase 5)

Trains a neural network to forecast the NIFTY implied-volatility surface 1, 5 and 20 trading days ahead,
and compares it **honestly** against two baselines on the same out-of-sample days:

* **No change** (random walk): tomorrow's surface = today's
* **PCA-VAR**: 3-factor PCA of the surface with a VAR(1) on the factors

**Before you start:** `Runtime > Change runtime type > T4 GPU`, then `Runtime > Run all`.
You will be asked to upload two files from your project folder:
`data/surfaces/NIFTY_grid.npz` and `data/surfaces/NIFTY_diagnostics.csv.gz`.

Full run takes roughly 15-30 minutes on a T4. Set `QUICK = True` below for a ~3 minute smoke test.

In [ ]:
QUICK = False            # True = fewer epochs/seeds/test years, just to check everything runs
USE_DRIVE = False        # True = read files from Google Drive folder below instead of uploading
DRIVE_DIR = "/content/drive/MyDrive/NEURAVOLAFORECAST"

HORIZONS = [1, 5, 20]
LOOKBACK = 20            # days of history the model sees
TEST_YEARS = list(range(2020, 2027)) if not QUICK else [2024]
SEEDS = [0, 1, 2] if not QUICK else [0]
MAX_EPOCHS = 150 if not QUICK else 15
PATIENCE = 20
COVID = ("2020-02-15", "2020-06-30")

In [ ]:
import os, json, time, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB and USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_DIR = DRIVE_DIR
elif IN_COLAB:
    from google.colab import files
    print("Upload NIFTY_grid.npz and NIFTY_diagnostics.csv.gz")
    files.upload()
    DATA_DIR = "/content"
else:
    DATA_DIR = os.environ.get("NEURAVOL_DATA", ".")
OUT_DIR = os.path.join(DATA_DIR if not IN_COLAB else "/content", "nn_outputs")
os.makedirs(OUT_DIR, exist_ok=True)

## 1. Load surfaces and build features

In [ ]:
z = np.load(os.path.join(DATA_DIR, "NIFTY_grid.npz"))
dates = pd.to_datetime(z["dates"])
IV = z["iv"].astype(np.float64)            # (days, tenors, moneyness), decimals
MASK = z["mask"].astype(bool)              # True = backed by real quotes
TENORS = z["tenors"].astype(float)
MON = z["moneyness"].astype(float)
D, NT, NK = IV.shape
NC = NT * NK
TAU = np.repeat(TENORS / 365.0, NK)        # maturity of each flattened cell

diag = pd.read_csv(os.path.join(DATA_DIR, "NIFTY_diagnostics.csv.gz"), parse_dates=["date"]).set_index("date")
diag = diag.reindex(dates)
fwd = diag["F_near"].ffill().values
ret = np.r_[0.0, np.diff(np.log(fwd))]
ret = np.clip(ret, -0.15, 0.15)            # near-forward roll jumps are tiny; clip glitches
rs = pd.Series(ret)
EXO = np.c_[
    ret * 100,
    np.abs(ret) * 100,
    rs.rolling(5, min_periods=1).std().fillna(0).values * np.sqrt(252) * 100,
    rs.rolling(21, min_periods=1).std().fillna(0).values * np.sqrt(252) * 100,
]
NE = EXO.shape[1]
LOGIV = np.log(IV.reshape(D, NC))
MASKF = MASK.reshape(D, NC)
print(f"{D} days {dates[0].date()} -> {dates[-1].date()}, grid {NT}x{NK}, exogenous features {NE}")

## 2. Scoring helpers and baselines

In [ ]:
def rmse_vp(pred_iv, true_iv, mask):
    """RMSE in vol points on cells backed by real quotes."""
    e = (pred_iv - true_iv)[mask] * 100
    return float(np.sqrt(np.mean(e ** 2))) if e.size else float("nan")


def daily_loss(pred_iv, true_iv, mask):
    se = ((pred_iv - true_iv) * 100) ** 2
    return np.array([se[i][mask[i]].mean() if mask[i].any() else np.nan for i in range(len(se))])


def diebold_mariano(loss_a, loss_b, h):
    """One-sided DM test that A beats B (Newey-West, h-1 lags). Returns (stat, p)."""
    d = np.asarray(loss_a) - np.asarray(loss_b)
    d = d[~np.isnan(d)]
    n = len(d)
    if n < 30:
        return float("nan"), float("nan")
    var = np.var(d)
    for l in range(1, max(h - 1, 0) + 1):
        var += 2 * (1 - l / h) * np.cov(d[l:], d[:-l], ddof=0)[0, 1]
    stat = d.mean() / np.sqrt(var / n)
    return float(stat), float(norm.cdf(stat))


def region_table(pred_iv, true_iv, mask):
    P, T_, M = (a.reshape(-1, NT, NK) for a in (pred_iv, true_iv, mask))
    out = {"overall": rmse_vp(pred_iv, true_iv, mask)}
    for i, t in enumerate(TENORS):
        out[f"{int(t)}d"] = rmse_vp(P[:, i], T_[:, i], M[:, i])
    for name, sel in {"put wing": MON <= -0.05, "near ATM": np.abs(MON) <= 0.025, "call wing": MON >= 0.05}.items():
        out[name] = rmse_vp(P[:, :, sel], T_[:, :, sel], M[:, :, sel])
    return out


def fit_pca_var(X, k=3):
    mu = X.mean(0)
    _, _, Vt = np.linalg.svd(X - mu, full_matrices=False)
    V = Vt[:k].T
    F = (X - mu) @ V
    B = np.linalg.lstsq(np.c_[np.ones(len(F) - 1), F[:-1]], F[1:], rcond=None)[0]
    return mu, V, B


def pca_var_forecast(x, model, h):
    mu, V, B = model
    f = (x - mu) @ V
    for _ in range(h):
        f = B[0] + f @ B[1:]
    return mu + f @ V.T


def test_indices(year, h):
    """Forecast origins t in `year` whose target t+h exists, and the first such index."""
    idx = np.where((dates.year == year) & (np.arange(D) >= LOOKBACK - 1) & (np.arange(D) + h < D))[0]
    return idx


IVF = IV.reshape(D, NC)


def baselines(h, years):
    """RW and PCA-VAR (refit every 21 days, expanding window) on the given test years."""
    origins = np.concatenate([test_indices(y, h) for y in years])
    rw = IVF[origins]
    pv = np.empty_like(rw)
    model, last_fit = None, -10**9
    for j, t in enumerate(origins):
        if t - last_fit >= 21:
            model, last_fit = fit_pca_var(IVF[: t + 1]), t   # surfaces up to today are known
        pv[j] = pca_var_forecast(IVF[t], model, h)
    return origins, {"No change": rw, "PCA-VAR": pv}

In [ ]:
# quick check: baselines on the test period
for h in HORIZONS:
    o, b = baselines(h, TEST_YEARS)
    tgt, msk = IVF[o + h], MASKF[o + h]
    print(f"h={h:2d}: " + ", ".join(f"{k} {rmse_vp(v, tgt, msk):.2f}" for k, v in b.items()), f"({len(o)} forecasts)")

## 3. The neural model

* Input: the last `LOOKBACK` surfaces (log-IV, 63 cells each) plus returns / realised-vol features.
* Encoder MLP per day, then a GRU over time, then a head that sees the GRU state **and** today's surface.
* Output: the **change** in log-IV for every cell. The last layer starts at zero, so an untrained
  model equals the "no change" forecast: the network only has to learn the deviation.
* Loss: error in vol points (cells backed by quotes weigh 1.0, extrapolated cells 0.1)
  **plus a calendar no-arbitrage penalty**: total variance must not fall as maturity grows.

In [ ]:
import torch
import torch.nn as nn

DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEV, torch.cuda.get_device_name(0) if DEV.type == "cuda" else "")
LAMBDA_ARB = 1.0
TAU_T = torch.tensor(TAU, dtype=torch.float32, device=DEV)


class SurfaceNet(nn.Module):
    def __init__(self, n_cells=NC, n_exo=NE, hid=128, drop=0.2):
        super().__init__()
        self.enc = nn.Sequential(nn.Linear(n_cells + n_exo, hid), nn.GELU(), nn.Dropout(drop),
                                 nn.Linear(hid, hid), nn.GELU())
        self.gru = nn.GRU(hid, hid, batch_first=True)
        self.head = nn.Sequential(nn.Dropout(drop), nn.Linear(hid + n_cells, hid), nn.GELU(), nn.Linear(hid, n_cells))
        nn.init.zeros_(self.head[-1].weight)
        nn.init.zeros_(self.head[-1].bias)

    def forward(self, seq, last):
        h, _ = self.gru(self.enc(seq))
        return self.head(torch.cat([h[:, -1], last], dim=1))


def make_stats(train_idx, h):
    """Standardisation stats from training origins only (no look-ahead)."""
    rows = np.unique(np.concatenate([train_idx - k for k in range(LOOKBACK)]))
    feats = np.c_[LOGIV, EXO][rows]
    dy = LOGIV[train_idx + h] - LOGIV[train_idx]
    return dict(mu=feats.mean(0), sd=feats.std(0) + 1e-6, ysd=dy.std(0) + 1e-4)


def tensors(idx, h, st):
    S = (np.c_[LOGIV, EXO] - st["mu"]) / st["sd"]
    seq = S[idx[:, None] - np.arange(LOOKBACK - 1, -1, -1)[None, :]]          # (B, L, NC+NE)
    last = S[idx, :NC]
    t = lambda a: torch.tensor(np.asarray(a), dtype=torch.float32, device=DEV)
    out = dict(seq=t(seq), last=t(last), logiv=t(LOGIV[idx]), ysd=t(st["ysd"]))
    if (idx + h < D).all():
        out["true"] = t(IVF[idx + h])
        out["w"] = t(np.where(MASKF[idx + h], 1.0, 0.1))
    return out


def predict_iv(model, b):
    return torch.exp(b["logiv"] + model(b["seq"], b["last"]) * b["ysd"])


def loss_fn(iv_pred, b, arb=True):
    err = (iv_pred - b["true"]) * 100
    loss = (b["w"] * err ** 2).sum() / b["w"].sum()
    if arb:
        wt = (iv_pred ** 2 * TAU_T).view(-1, NT, NK)
        loss = loss + LAMBDA_ARB * 1e4 * torch.relu(wt[:, :-1] - wt[:, 1:]).sum(dim=(1, 2)).mean()
    return loss


def subset(b, i):
    return {k: (v[i] if k != "ysd" else v) for k, v in b.items()}


def fit_one(train_idx, val_idx, h, st, seed, bs=64, lr=1e-3):
    torch.manual_seed(seed); np.random.seed(seed)
    model = SurfaceNet().to(DEV)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    tr, va = tensors(train_idx, h, st), tensors(val_idx, h, st)
    best, best_state, wait = float("inf"), None, 0
    n = len(train_idx)
    for ep in range(MAX_EPOCHS):
        model.train()
        perm = torch.randperm(n, device=DEV)
        for s in range(0, n, bs):
            b = subset(tr, perm[s:s + bs])
            loss = loss_fn(predict_iv(model, b), b)
            opt.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
        model.eval()
        with torch.no_grad():
            vl = loss_fn(predict_iv(model, va), va, arb=False).item()
        if vl < best - 1e-4:
            best, wait = vl, 0
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        else:
            wait += 1
            if wait >= PATIENCE:
                break
    model.load_state_dict(best_state)
    model.eval()
    return model, ep + 1, math.sqrt(best)


def split_train_val(last_origin_exclusive, h, val_frac=0.15):
    """Chronological train/val split of origins whose targets are known before `last_origin_exclusive`."""
    usable = np.arange(LOOKBACK - 1, last_origin_exclusive - h)
    n_val = max(50, int(len(usable) * val_frac))
    val = usable[-n_val:]
    train = usable[usable + h < val[0]]          # embargo: no train target overlaps validation
    return train, val


def ensemble_predict(models, idx, h, st):
    b = tensors(idx, h, st)
    with torch.no_grad():
        logs = torch.stack([torch.log(predict_iv(m, b)) for m in models]).mean(0)
    return torch.exp(logs).cpu().numpy()

## 4. Walk-forward evaluation
For each test year: train on everything before it (with a validation slice for early stopping),
forecast every day of that year, then move on. Same forecast days as the baselines.

In [ ]:
results, all_preds = {}, {}
t0 = time.time()
for h in HORIZONS:
    origins, base = baselines(h, TEST_YEARS)
    nn_pred = np.empty_like(base["No change"])
    for y in TEST_YEARS:
        test = test_indices(y, h)
        if len(test) == 0:
            continue
        train, val = split_train_val(test[0], h)
        st = make_stats(train, h)
        models, info = [], []
        for seed in SEEDS:
            m, ep, vrmse = fit_one(train, val, h, st, seed)
            models.append(m); info.append(f"{ep}ep/{vrmse:.2f}")
        nn_pred[np.isin(origins, test)] = ensemble_predict(models, test, h, st)
        print(f"h={h:2d} {y}: train {len(train)} val {len(val)} test {len(test)} | seeds {info} | {time.time() - t0:.0f}s")
    preds = {**base, "NeuraVol NN": nn_pred}
    tgt, msk = IVF[origins + h], MASKF[origins + h]
    calm = ~((dates[origins] >= COVID[0]) & (dates[origins] <= COVID[1]))
    tab = pd.DataFrame({k: region_table(v, tgt, msk) for k, v in preds.items()})
    tab.loc["excl. COVID crash"] = [rmse_vp(v[calm], tgt[calm], msk[calm]) for v in preds.values()]
    losses = {k: daily_loss(v, tgt, msk) for k, v in preds.items()}
    dm = {f"NN vs {k}": diebold_mariano(losses["NeuraVol NN"], losses[k], h) for k in base}
    arb = (((preds["NeuraVol NN"] ** 2 * TAU).reshape(-1, NT, NK)[:, :-1] -
            (preds["NeuraVol NN"] ** 2 * TAU).reshape(-1, NT, NK)[:, 1:]) > 1e-8).mean()
    results[h] = dict(table=tab, dm=dm, calendar_violation_rate=float(arb), n=len(origins))
    all_preds[h] = dict(origins=origins, target=tgt, mask=msk, **preds)
    print(f"\n=== {h}-day horizon: RMSE in vol points ({len(origins)} forecasts) ===")
    print(tab.round(2).to_string())
    for k, (s, p) in dm.items():
        print(f"Diebold-Mariano {k}: stat {s:.2f}, p = {p:.3f}  {'<- NN significantly better' if p < 0.05 else ''}")
    print(f"NN calendar-arbitrage violations: {arb:.2%} of cell pairs\n")

## 5. Results

In [ ]:
summary = pd.DataFrame({f"{h}d": r["table"].loc["overall"] for h, r in results.items()}).T
summary_calm = pd.DataFrame({f"{h}d": r["table"].loc["excl. COVID crash"] for h, r in results.items()}).T
print("Overall RMSE (vol points)\n", summary.round(2), "\n\nExcluding Feb-Jun 2020\n", summary_calm.round(2))

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
summary.plot.bar(ax=ax[0], rot=0, title="Forecast error, all test days (lower is better)")
summary_calm.plot.bar(ax=ax[1], rot=0, title="Forecast error, excluding COVID crash")
for a in ax:
    a.set_ylabel("RMSE, vol points")
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, "rmse_by_horizon.png"), dpi=130); plt.show()

h = 5 if 5 in all_preds else HORIZONS[0]
P = all_preds[h]
c = int(np.where(TENORS == 30)[0][0]) * NK + int(np.argmin(np.abs(MON)))
d = dates[P["origins"] + h]
plt.figure(figsize=(13, 4))
plt.plot(d, P["target"][:, c] * 100, lw=1, label="actual")
plt.plot(d, P["NeuraVol NN"][:, c] * 100, lw=1, label=f"NN forecast made {h} days earlier")
plt.plot(d, P["No change"][:, c] * 100, lw=0.8, alpha=0.6, label="no-change forecast")
plt.title("30-day ATM implied vol"); plt.ylabel("IV %"); plt.legend()
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, "atm30_forecast_vs_actual.png"), dpi=130); plt.show()

with open(os.path.join(OUT_DIR, "nn_results.json"), "w") as f:
    json.dump({str(h): dict(table=r["table"].round(4).to_dict(), dm=r["dm"],
                            calendar_violation_rate=r["calendar_violation_rate"], n=r["n"])
               for h, r in results.items()}, f, indent=2)
np.savez_compressed(os.path.join(OUT_DIR, "nn_test_predictions.npz"),
                    **{f"h{h}_{k.replace(' ', '_')}": v for h, P in all_preds.items() for k, v in P.items()})

## 6. Train the final models on all data and forecast from the latest day
These weights are what the deployed dashboard will load (CPU is enough to run them).

In [ ]:
bundle = {"tenors": TENORS.tolist(), "moneyness": MON.tolist(), "lookback": LOOKBACK,
          "last_date": str(dates[-1].date()), "models": {}}
latest = np.array([D - 1])
rows = []
for h in HORIZONS:
    train, val = split_train_val(D, h, val_frac=0.10)
    st = make_stats(train, h)
    models = [fit_one(train, val, h, st, seed)[0] for seed in SEEDS]
    bundle["models"][h] = {"stats": {k: v.tolist() for k, v in st.items()},
                           "state_dicts": [{k: v.cpu() for k, v in m.state_dict().items()} for m in models]}
    fc = ensemble_predict(models, latest, h, st)[0].reshape(NT, NK)
    for i, t in enumerate(TENORS):
        for j, k in enumerate(MON):
            rows.append(dict(as_of=dates[-1].date(), horizon_days=h, tenor_days=int(t), moneyness=k,
                             iv_today=IV[-1, i, j], iv_forecast=fc[i, j], backed_by_quotes=bool(MASK[-1, i, j])))
    print(f"h={h}: final model trained on {len(train)} days")
torch.save(bundle, os.path.join(OUT_DIR, "neuravol_models.pt"))
fc_df = pd.DataFrame(rows)
fc_df.to_csv(os.path.join(OUT_DIR, "latest_forecast.csv"), index=False)
print(fc_df[(fc_df.moneyness == 0)].round(4).to_string(index=False))

In [ ]:
if IN_COLAB:
    import shutil
    from google.colab import files
    shutil.make_archive("/content/nn_outputs", "zip", OUT_DIR)
    files.download("/content/nn_outputs.zip")
    print("Downloaded nn_outputs.zip: unzip it into your project's reports/ folder")